# FEMTO PHM 2012 — Veri Keşfi

Bu notebook üç amaca hizmet eder:
1. Veri setini tanıma: bearing'lerin yaşam süreleri, RUL dağılımı
2. Öznitelikleri anlama: hangi öznitelikler bozunumla nasıl değişiyor
3. Modelleme öncesi kontrol: train/val setlerinin benzerliği

**Kaynak:** `data/processed/features_train.csv` ve `features_test.csv`

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from data_utils import get_feature_cols, train_val_split
from config import FEATURES_TRAIN, FEATURES_TEST

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='tab10')

# Veriyi yükle (config.py'deki mutlak path'ler kullanılır)
df_train = pd.read_csv(FEATURES_TRAIN)
df_test  = pd.read_csv(FEATURES_TEST)
feature_cols = get_feature_cols(df_train)

print(f'Learning set: {df_train.shape[0]} pencere, {len(feature_cols)} öznitelik')
print(f'Test set    : {df_test.shape[0]} pencere')
print(f'\nBearing\'ler: {sorted(df_train["bearing"].unique())}')

---
## 1. Bearing Yaşam Süreleri

In [1]:
# Her bearing için özet bilgi
summary = df_train.groupby('bearing').agg(
    pencere_sayisi=('window_idx', 'count'),
    toplam_omur_dk=('time_s', lambda x: (x.max() + 10) / 60),
    max_rul_dk=('rul_min', 'max'),
    min_rul_dk=('rul_min', 'min'),
    calisma_kosulu=('condition', 'first')
).reset_index()

print('=== Learning Set Özeti ===')
print(summary.to_string(index=False))

NameError: name 'df_train' is not defined

In [ ]:
# Çalışma koşuluna göre renklendir
cond_colors = {1: '#2196F3', 2: '#4CAF50', 3: '#FF5722'}
cond_labels = {1: 'Koşul 1 (1800rpm/4000N)', 2: 'Koşul 2 (1650rpm/4200N)', 3: 'Koşul 3 (1500rpm/5000N)'}

fig, ax = plt.subplots(figsize=(10, 4))
bearings = summary['bearing'].tolist()
lifetimes = summary['toplam_omur_dk'].tolist()
colors = [cond_colors[c] for c in summary['calisma_kosulu']]

bars = ax.barh(bearings, lifetimes, color=colors, edgecolor='white', height=0.6)
for bar, val in zip(bars, lifetimes):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f} dk', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=cond_colors[c], label=cond_labels[c]) for c in [1,2,3]]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

ax.set_xlabel('Toplam Ömür (dakika)')
ax.set_title('Learning Set — Bearing Yaşam Süreleri', fontweight='bold')
ax.axvline(125, color='red', linestyle='--', linewidth=1.2, label='RUL Cap (125 dk)')
plt.tight_layout()
plt.show()

print('\nNot: Kırmızı kesikli çizgi = RUL tavan değeri (125 dk)')
print('Bearing3_1 toplam ömrü < 125 dk olduğundan tavan etkisiz kalmıştır.')

Her bir rulman için inceleme

In [ ]:
# Her bearing için RMS zaman serisi — bozulma başlangıcını görmek için
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, bname in zip(axes, sorted(df_train['bearing'].unique())):
    bdf = df_train[df_train['bearing'] == bname].sort_values('time_s')
    time_min = bdf['time_s'] / 60
    rms = bdf['h_rms']
    smoothed = rms.rolling(30, min_periods=1).mean()
    total = time_min.max()
    
    ax.plot(time_min, rms, alpha=0.2, color='gray', linewidth=0.5)
    ax.plot(time_min, smoothed, color='#2196F3', linewidth=2)
    
    # Bozulma başlangıcı: RMS ilk kez baseline'ın 2 katını geçtiği an
    baseline = rms.iloc[:50].mean()
    threshold = baseline * 2
    degradation_start = bdf[rms > threshold]['time_s']
    if len(degradation_start) > 0:
        t_deg = degradation_start.iloc[0] / 60
        ax.axvline(t_deg, color='orange', linestyle='--', linewidth=1.5,
                   label=f'Bozulma başlangıcı: {t_deg:.1f} dk')
        ax.axvline(total, color='red', linestyle='-', linewidth=1.5,
                   label=f'Arıza: {total:.1f} dk')
        remaining = total - t_deg
        ax.set_title(f'{bname}\nBozulma süresi: {remaining:.1f} dk', 
                     fontweight='bold', fontsize=9)
    else:
        ax.set_title(bname, fontweight='bold', fontsize=9)
    
    ax.legend(fontsize=7)
    ax.set_xlabel('Süre (dk)', fontsize=8)
    ax.set_ylabel('RMS', fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Her Bearing için RMS Bozulma Paterni\n(Turuncu = bozulma başlangıcı, Kırmızı = arıza)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, bname in zip(axes, sorted(df_train['bearing'].unique())):
    bdf = df_train[df_train['bearing'] == bname].sort_values('time_s')
    time_min = bdf['time_s'] / 60
    total = time_min.max()

    rms_s  = bdf['h_rms'].rolling(30, min_periods=1).mean()
    kurt_s = bdf['h_kurtosis'].rolling(30, min_periods=1).mean()

    ax2 = ax.twinx()  # ikinci y ekseni

    ax.plot(time_min, rms_s,  color='#2196F3', linewidth=2, label='RMS')
    ax2.plot(time_min, kurt_s, color='#FF5722', linewidth=1.5,
             linestyle='--', label='Kurtosis', alpha=0.8)

    ax.axvline(total, color='red', linewidth=1.5)
    ax.set_title(bname, fontweight='bold', fontsize=9)
    ax.set_xlabel('Süre (dk)', fontsize=8)
    ax.set_ylabel('RMS', color='#2196F3', fontsize=8)
    ax2.set_ylabel('Kurtosis', color='#FF5722', fontsize=8)
    ax.grid(True, alpha=0.3)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=7)

plt.suptitle('RMS vs Kurtosis — Bozulma Tespiti Karşılaştırması',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2. RUL Etiketleri — Piecewise Lineer Yapı

In [ ]:
# Her bearing için RUL zaman serisi
bearings = sorted(df_train['bearing'].unique())
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=False)
axes = axes.flatten()

for ax, bname in zip(axes, bearings):
    bdf = df_train[df_train['bearing'] == bname].sort_values('time_s')
    cond = bdf['condition'].iloc[0]
    color = cond_colors[cond]

    ax.plot(bdf['time_s'] / 60, bdf['rul_min'],
            color=color, linewidth=1.8)
    ax.fill_between(bdf['time_s'] / 60, bdf['rul_min'],
                    alpha=0.15, color=color)
    ax.set_title(f'{bname} (Koşul {cond})', fontsize=10, fontweight='bold')
    ax.set_xlabel('Geçen Süre (dk)', fontsize=8)
    ax.set_ylabel('RUL (dk)', fontsize=8)
    ax.axhline(0, color='black', linewidth=0.8, linestyle=':')
    ax.grid(True, alpha=0.3)

plt.suptitle('Piecewise Lineer RUL Etiketleri — Learning Set',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Düz yatay çizgi = RUL cap dönemi (erken evre, model bu dönemi "uzun kalan ömür" olarak öğrenir)')
print('Azalan eğim    = Gerçek kalan ömür (modelin tahmin etmesi gereken bölge)')

---
## 3. Özniteliklerin Bozunum Boyunca Değişimi

En çok incelenen öznitelik: **RMS** ve **Kurtosis** — bozunumla nasıl değişiyor?

In [ ]:
# Bearing1_1 üzerinde tüm temel özniteliklerin zaman serisi
bdf = df_train[df_train['bearing'] == 'Bearing1_1'].sort_values('time_s')
time_min = bdf['time_s'] / 60

plot_features = [
    ('h_rms',           'RMS (Yatay)',          '#2196F3'),
    ('v_rms',           'RMS (Dikey)',           '#03A9F4'),
    ('h_kurtosis',      'Kurtosis (Yatay)',      '#F44336'),
    ('v_kurtosis',      'Kurtosis (Dikey)',      '#FF7043'),
    ('h_crest_factor',  'Crest Factor (Yatay)',  '#9C27B0'),
    ('h_spectral_entropy', 'Spektral Entropi (Yatay)', '#009688'),
]

fig, axes = plt.subplots(3, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, (feat, label, color) in zip(axes, plot_features):
    # Ham sinyal (gri, ince)
    ax.plot(time_min, bdf[feat], color=color, alpha=0.3, linewidth=0.7)
    # Hareketli ortalama (kalın, temiz görünüm)
    smoothed = bdf[feat].rolling(window=50, min_periods=1).mean()
    ax.plot(time_min, smoothed, color=color, linewidth=2, label='Hareketli ort. (50)')
    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_xlabel('Süre (dk)', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)

plt.suptitle('Bearing1_1 — Özniteliklerin Zamanla Değişimi',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Tüm bearing'lerde RMS karşılaştırması (normalize edilmiş zaman ekseni)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for bname in bearings:
    bdf = df_train[df_train['bearing'] == bname].sort_values('time_s')
    # Normalize zaman: 0 (başlangıç) → 1 (arıza)
    norm_time = bdf['time_s'] / bdf['time_s'].max()
    cond = bdf['condition'].iloc[0]
    color = cond_colors[cond]

    smoothed_rms  = bdf['h_rms'].rolling(30, min_periods=1).mean()
    smoothed_kurt = bdf['h_kurtosis'].rolling(30, min_periods=1).mean()

    axes[0].plot(norm_time, smoothed_rms,
                 color=color, linewidth=1.5, alpha=0.8, label=bname)
    axes[1].plot(norm_time, smoothed_kurt,
                 color=color, linewidth=1.5, alpha=0.8, label=bname)

for ax, title in zip(axes, ['RMS (Yatay)', 'Kurtosis (Yatay)']):
    ax.set_xlabel('Normalize Yaşam (0=başlangıç, 1=arıza)', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('Tüm Bearing\'lerde Öznitelik Değişimi (Normalize Zaman)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print('RMS: Bozunum ilerledikçe genellikle artar — iyi bir RUL göstergesi')
print('Kurtosis: Erken hasarda zirve yapar, ileri hasarda düşebilir (U-şekli beklenir)')

---
## 4. Öznitelik — RUL Korelasyonu

In [ ]:
# Her özniteliğin RUL ile Pearson korelasyonu
corr_vals = df_train[feature_cols + ['rul_min']].corr()['rul_min'].drop('rul_min')
corr_sorted = corr_vals.abs().sort_values(ascending=False)

# En yüksek 15 ve en düşük 5
top15 = corr_vals[corr_sorted.index[:15]]

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = ['#F44336' if v < 0 else '#2196F3' for v in top15.values]
bars = ax.barh(top15.index[::-1], top15.values[::-1],
               color=colors_bar[::-1], edgecolor='white', height=0.7)

ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Korelasyon Katsayısı (RUL ile)', fontsize=10)
ax.set_title('En Yüksek RUL Korelasyonuna Sahip 15 Öznitelik',
             fontweight='bold', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')

for bar, val in zip(bars, top15.values[::-1]):
    ax.text(val + (0.005 if val >= 0 else -0.005),
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center',
            ha='left' if val >= 0 else 'right', fontsize=8)

plt.tight_layout()
plt.show()

print('\nMavi = pozitif korelasyon (öznitelik azaldıkça RUL de azalır)')
print('Kırmızı = negatif korelasyon (öznitelik arttıkça RUL azalır = bozunum göstergesi)')

In [ ]:
# Öznitelikler arası korelasyon ısı haritası (multicollinearity kontrolü)
# Sadece seçili önemli öznitelikler
selected = [
    'h_rms', 'h_kurtosis', 'h_crest_factor', 'h_skewness',
    'h_spectral_entropy', 'h_band_energy_1', 'h_band_energy_2',
    'v_rms', 'v_kurtosis', 'cross_corr', 'rul_min'
]

corr_matrix = df_train[selected].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Öznitelik Korelasyon Matrisi', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.show()

print('Yüksek korelasyonlu öznitelik çiftleri modelde redundant bilgi taşır.')
print('Örn: h_rms ve v_rms genellikle yüksek korelasyon gösterir.')

---
## 5. Train / Validation Set Karşılaştırması

In [ ]:
train_df, val_df = train_val_split(df_train)

# RUL dağılımı karşılaştırması
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Sol: RUL histogram
axes[0].hist(train_df['rul_min'], bins=50, alpha=0.6,
             color='#2196F3', label=f'Train ({len(train_df)} pencere)')
axes[0].hist(val_df['rul_min'],   bins=50, alpha=0.6,
             color='#FF5722', label=f'Val ({len(val_df)} pencere)')
axes[0].set_xlabel('RUL (dakika)')
axes[0].set_ylabel('Pencere Sayısı')
axes[0].set_title('RUL Dağılımı: Train vs Val', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Sağ: Koşul dağılımı pasta grafik
train_cond = train_df.groupby('condition')['window_idx'].count()
val_cond   = val_df.groupby('condition')['window_idx'].count()

x = np.arange(len(train_cond))
w = 0.35
axes[1].bar(x - w/2, train_cond.values, w, label='Train',
            color='#2196F3', alpha=0.8)
axes[1].bar(x + w/2, val_cond.values,   w, label='Val',
            color='#FF5722', alpha=0.8)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'Koşul {c}' for c in train_cond.index])
axes[1].set_ylabel('Pencere Sayısı')
axes[1].set_title('Çalışma Koşulu Dağılımı', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Seçili özniteliklerin train/val istatistik karşılaştırması
check_feats = ['h_rms', 'h_kurtosis', 'h_crest_factor', 'h_spectral_entropy']

print(f'{'Öznitelik':<25} {'Train Mean':>12} {'Train Std':>11} {'Val Mean':>10} {'Val Std':>9}')
print('-' * 70)
for feat in check_feats:
    tm, ts = train_df[feat].mean(), train_df[feat].std()
    vm, vs = val_df[feat].mean(),   val_df[feat].std()
    print(f'{feat:<25} {tm:>12.4f} {ts:>11.4f} {vm:>10.4f} {vs:>9.4f}')

print('\nTrain ve Val istatistikleri birbirine yakınsa split dengeli demektir.')

---
## 6. Bozunum Evrelerinde Öznitelik Kutu Grafikleri

In [ ]:
# RUL'u 4 evreye böl ve öznitelik dağılımını karşılaştır
df_train = df_train.copy()
df_train['evre'] = pd.cut(
    df_train['rul_min'],
    bins=[0, 20, 50, 90, 125.1],
    labels=['Kritik\n(0-20dk)', 'Geç\n(20-50dk)', 'Orta\n(50-90dk)', 'Erken\n(90-125dk)']
)

plot_feats = ['h_rms', 'h_kurtosis', 'h_crest_factor', 'h_spectral_entropy']
feat_labels = ['RMS (Yatay)', 'Kurtosis (Yatay)', 'Crest Factor (Yatay)', 'Spektral Entropi']

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.flatten()

for ax, feat, label in zip(axes, plot_feats, feat_labels):
    data_by_evre = [
        df_train[df_train['evre'] == e][feat].dropna().values
        for e in ['Erken\n(90-125dk)', 'Orta\n(50-90dk)',
                  'Geç\n(20-50dk)', 'Kritik\n(0-20dk)']
    ]
    bp = ax.boxplot(data_by_evre,
                    labels=['Erken\n(90-125dk)', 'Orta\n(50-90dk)',
                            'Geç\n(20-50dk)', 'Kritik\n(0-20dk)'],
                    patch_artist=True,
                    medianprops=dict(color='red', linewidth=2))

    colors_box = ['#81D4FA', '#4FC3F7', '#0288D1', '#01579B']
    for patch, c in zip(bp['boxes'], colors_box):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)

    ax.set_title(label, fontweight='bold', fontsize=10)
    ax.set_ylabel(feat)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Bozunum Evrelerine Göre Öznitelik Dağılımı',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Kutular soldan sağa: Erken evre → Kritik evre')
print('Evreler arasında belirgin fark = öznitelik RUL tahmini için değerli')

---
## 7. Özet Bulgular

Bu hücreyi çalıştırdıktan sonra tez için notlar alabileceğin bir özet.

In [ ]:
print('=' * 55)
print('  VERİ KEŞFİ — ÖZET BULGULAR')
print('=' * 55)

print(f'\n• Toplam learning penceresi : {len(df_train)}')
print(f'• Toplam öznitelik sayısı  : {len(feature_cols)}')
print(f'• Bearing yaşam aralığı    : {df_train.groupby("bearing")["time_s"].max().min()/60:.1f} — '
      f'{df_train.groupby("bearing")["time_s"].max().max()/60:.1f} dakika')

# En güçlü 5 öznitelik
top5 = corr_vals.abs().sort_values(ascending=False).head(5)
print(f'\n• RUL ile en yüksek korelasyonlu 5 öznitelik:')
for feat, val in top5.items():
    direction = '+' if corr_vals[feat] > 0 else '-'
    print(f'    {feat:<30} r = {direction}{val:.3f}')

print('\n• Piecewise RUL cap (125 dk): Bearing3_1 hariç tüm bearing\'lerde tavan aktif')
print('• Train/Val split  : 3 bearing train, 3 bearing val (bearing bazlı)')